# Compare fp16/fp32 diff

In [1]:
import torch
from pathlib import Path
import os

In [2]:
path_fp16 = Path(f"/home/siddhartht/tts/speechLM/NeMo_2503/layerwise_output/torch.float16/")
path_fp32 = Path(f"/home/siddhartht/tts/speechLM/NeMo_2503/layerwise_output/torch.float32/")

In [76]:
diff = {}
for layer_ in path_fp16.iterdir():
    fp_16_path = layer_
    fp_32_path = path_fp32 / layer_.name
    
    lo_fp16 = torch.load(fp_16_path)
    lo_fp32 = torch.load(fp_32_path)
    
    eps = 1e-7
    if "dec_out" in layer_.name or "attn_probs" in layer_.name:
        for idx, sub_l_fp16 in enumerate(lo_fp16):
            if "attn_probs" in layer_.name:
                sub_l_fp32 = lo_fp32[idx]["self_attn_probabilities"][0]
                sub_l_fp16 = sub_l_fp16["self_attn_probabilities"][0]
                
                sum_diff = sub_l_fp32 - sub_l_fp16.float()
                sum_diff = sum_diff.abs()/(sub_l_fp32+eps).abs().sum()
                sum_diff = sum_diff.sum() #/sum_diff.numel()
                diff[f"{layer_.stem}.{idx}.self_attn"] = {"avg": sum_diff.mean().item(),
                                                        "median": sum_diff.median().item(),
                                                       "max": sum_diff.max().item()}
                
                sub_l_fp32 = lo_fp32[idx]["cross_attn_probabilities"][0]
                sub_l_fp16 = lo_fp16[idx]["cross_attn_probabilities"][0]
                
                sum_diff = sub_l_fp32 - sub_l_fp16.float()
                sum_diff = sum_diff.abs()/(sub_l_fp32+eps).abs()
                #sum_diff = sum_diff.sum() #/sum_diff.numel()
                diff[f"{layer_.stem}.{idx}.cross_attn"] = {"avg": sum_diff.mean().item(),
                                                        "median": sum_diff.median().item(),
                                                       "max": sum_diff.max().item()}
            else:
                sub_l_fp32 = lo_fp32[idx]
                print(layer_, sub_l_fp16.shape, sub_l_fp32.shape)
                sum_diff = sub_l_fp32 - sub_l_fp16.float()
                sum_diff = sum_diff.abs()/sub_l_fp32.abs()
                diff[f"{layer_.stem}.{idx}.dec_out"] = {"avg": sum_diff.mean().item(),
                                                        "median": sum_diff.median().item(),
                                                       "max": sum_diff.max().item()}

    else:
        sum_diff = lo_fp32 - lo_fp16.float()
        sum_diff = sum_diff.abs()/lo_fp32.abs()
        #sum_diff = sum_diff.sum() #/sum_diff.numel()
        diff[f"{layer_.stem}"] = {
            "avg": sum_diff.mean().item(), "max": sum_diff.max().item(), "median": sum_diff.median().item()}

/home/siddhartht/tts/speechLM/NeMo_2503/layerwise_output/torch.float16/dec_out.pt torch.Size([110, 768]) torch.Size([110, 768])
/home/siddhartht/tts/speechLM/NeMo_2503/layerwise_output/torch.float16/dec_out.pt torch.Size([110, 768]) torch.Size([110, 768])


/tmp/ipykernel_2227379/4264348748.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lo_fp16 = torch.load(fp_16_path)
/tmp/ipykernel_2227379/4264348748.py:7: FutureWarning:

In [77]:
diff

{'audio_codes_embedded': {'avg': 0.0016856755828484893,
  'max': 0.11981412023305893,
  'median': 0.0005044136778451502},
 'combined_logits': {'avg': 0.008715703152120113,
  'max': 5523.36376953125,
  'median': 0.0009823982836678624},
 'attn_probs.0.self_attn': {'avg': 0.0005064749857410789,
  'median': 0.0005064749857410789,
  'max': 0.0005064749857410789},
 'attn_probs.0.cross_attn': {'avg': 0.0008791565778665245,
  'median': 0.0,
  'max': 0.023571090772747993},
 'attn_probs.1.self_attn': {'avg': 0.0005287499516271055,
  'median': 0.0005287499516271055,
  'max': 0.0005287499516271055},
 'attn_probs.1.cross_attn': {'avg': 0.0009210851858370006,
  'median': 0.0,
  'max': 0.008037587627768517},
 'attn_probs.2.self_attn': {'avg': 0.0008269196841865778,
  'median': 0.0008269196841865778,
  'max': 0.0008269196841865778},
 'attn_probs.2.cross_attn': {'avg': 0.007695264182984829,
  'median': 0.0,
  'max': 0.23555342853069305},
 'attn_probs.3.self_attn': {'avg': 0.0008366419933736324,
  'medi

In [80]:
import pandas as pd

df_diff = {"layer_out": list(diff.keys()),
           "avg": [i["avg"] for i in list(diff.values())],
           "max": [i["max"] for i in list(diff.values())],
           "median": [i["median"] for i in list(diff.values())]
          }
df = pd.DataFrame(df_diff)

In [82]:
df.sort_values(by="max", ascending=False)

,layer_out,avg,max,median
1,combined_logits,0.008716,5523.363770,0.000982
27,context_embeddings,0.030967,2278.098389,0.000612
29,dec_out.0.dec_out,0.018519,669.525085,0.001609
30,dec_out.1.dec_out,0.014962,312.373566,0.001310
31,text_encoder_out,0.003701,6.413437,0.000776
7,attn_probs.2.cross_attn,0.007695,0.235553,0.000000
9,attn_probs.3.cross_attn,0.014887,0.235479,0.000000
17,attn_probs.7.cross_attn,0.011962,0.231428,0.000000
11,attn_probs.4.cross_attn,0.006650,0.229161,0.000000
19,attn_probs.8.cross_attn,0.010589,0.228863,0.000000
